# [실습] RAG 파이프라인 성능 평가하기   

지금까지 RAG의 성능을 높이기 위한 다양한 방법에 대해 알아봤는데요.   
실제 RAG의 성능은 어떻게 측정해야 할까요?

이번 실습에서는 정답이 존재하는 RAG 데이터를 이용해, 성능을 평가하는 과정에 대해 알아보겠습니다.

### 라이브러리 설치  

랭체인 관련 라이브러리와 벡터 데이터베이스 라이브러리를 설치합니다.   

In [ ]:
!pip install sacrebleu ragas dotenv langchain_huggingface jsonlines langchain==0.3.27 langchain-openai langchain-community==0.3.27 beautifulsoup4 langchain_chroma

  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached MarkupSafe-3.0.2-cp313-cp313-win_amd64.whl.metadata (4.1 kB)
Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)

  Attempting uninstall: fsspec

    Found existing installation: fsspec 2025.7.0

    Uninstalling fsspec-2025.7.0:

   -------------- -------------------------  5/14 [fsspec]
      Successfully uninstalled fsspec-2025.7.0
   -------------- -------------------------  5/14 [fsspec]
   -------------- -------------------------  5/14 [fsspec]
   -------------------- -------------------  7/14 [diskcache]
   ------------------------- --------------  9/14 [jinja2]
   ---------------------------- ----------- 10/14 [gitdb]
   ------------------------------- -------- 11/14 [gitpython]
   ---------------------------------- ---

In [2]:
import os
from dotenv import load_dotenv
load_dotenv('.env', override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')

OpenAI API 키 확인


In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano", temperature = 0, max_tokens = 4096)
# 4o : Context 128k - 200k
# 4.1 : Context 1M
# 4o-mini > 4.1 Nano


RAG의 평가를 위해서는 정답이 있는 Q/A 데이터가 필요합니다.   
실습 시트에서 eval.jsonl을 다운로드하여 불러옵니다.

In [7]:
import pandas as pd

evaluation_dataset = pd.read_csv('./eval_dataset.csv', encoding='cp949')
evaluation_dataset

,questions,ground_truths
0,"2022~2024년 삼성SDS의 매출 구성을 IT서비스와 물류 관점에서 비교하고, ...",공시자료에 따르면 전체 매출에서 IT서비스와 물류 비중은 2022년 34.6% 대 ...
1,"삼성SDS가 ‘클라우드 기업’으로 전환한다고 밝힌 바 있는 전략을, 제품·플랫폼·인...",제품·플랫폼 측면에서 회사는 SCP를 전개하며 CSP·MSP·SaaS를 End-to...
2,"물류 부문에서 삼성SDS가 디지털 포워딩 경쟁력을 어떻게 구축했는지, 플랫폼 진화와...",삼성SDS는 통합물류 플랫폼 Cello와 디지털 물류 플랫폼 Cello Square...
3,"회사가 서비스 가격표를 제시하지 않는 이유와 실제 가격 결정 방식은 무엇이며, 이는...",공시자료는 IT서비스(SI·ITO·클라우드)와 물류가 개인용 규격 상품이 아니라 고...
4,"경기변동이 IT서비스와 물류에 미치는 영향과, 삼성SDS가 그 민감도를 낮추기 위해...","IT서비스는 기업 설비투자와 연동돼 경기 둔화 시 투자 보류 영향이 있으나, 클라우..."
5,SRM 분야에서의 인수·출시 활동이 삼성SDS의 SaaS 전략과 글로벌 확장에 어떤...,회사는 2023년 2분기에 국내 1위 SRM 기업 엠로의 지분 인수를 완료해 핵심 ...
6,인력·시설·보안 측면에서 삼성SDS의 운영 기반과 이것이 서비스 신뢰성에 주는 함의...,"2024년 말 기준 연결 인력은 약 25,666명으로, IT서비스 16,798명, ..."
7,"재무 안정성 측면에서 회사의 공시 내용을 종합하되, 상장 이력과 조달 구조까지 연결...","삼성SDS는 2014년 11월 14일 유가증권시장에 상장했으며, 단기 조달 수단인 ..."
8,삼성SDS의 2024년 3분기말(9월 30일) 연결 기준 부채비율을 재무상태표 수치...,"2024년 3분기말 연결 기준 부채총계는 3,409,626,256천원, 자본총계는 ..."
9,2024년 3분기 누적과 2024 사업연도(사업보고서 기준)에서 삼성SDS의 희석주...,삼성SDS는 당기 및 전기에 희석증권을 보유하지 않아 희석주당이익이 기본주당이익과 ...


기존 데이터를 불러옵니다.

In [ ]:
from glob import glob
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

pdf_files = glob("reports/*.pdf")
pdf_files

# 각 PDF 파일에서 페이지별로 내용을 불러와 하나로 합침
all_papers=[]

for i, path_paper in enumerate(pdf_files):
    loader = PyMuPDFLoader(path_paper)
    pages = loader.load()
    doc = Document(page_content='', metadata = {'index':i, 'source':pages[0].metadata['source']})
    for page in pages:
        doc.page_content += page.page_content +' '

    doc.page_content = doc.page_content.replace('\n', ' ')
    for _ in range(10):
        doc.page_content = doc.page_content.replace('  ', ' ')
        doc.page_content = doc.page_content.replace('..', '.')

    all_papers.append(doc)

print(len(all_papers))
all_papers[0].page_content[:1000]


3


'목 차 반 기 보 고 서.1 【 대표이사 등의 확인 】.2 I. 회사의 개요.3 1. 회사의 개요.3 2. 회사의 연혁.5 3. 자본금 변동사항.8 4. 주식의 총수 등.9 5. 정관에 관한 사항.10 II. 사업의 내용.12 1. 사업의 개요.12 2. 주요 제품 및 서비스.13 3. 원재료 및 생산설비.17 4. 매출 및 수주상황.19 5. 위험관리 및 파생거래.20 6. 주요계약 및 연구개발활동.22 7. 기타 참고사항.25 III. 재무에 관한 사항.34 1. 요약재무정보.34 2. 연결재무제표.37 2-1. 연결 재무상태표.37 2-2. 연결 손익계산서.38 2-3. 연결 포괄손익계산서.39 2-4. 연결 자본변동표.39 2-5. 연결 현금흐름표.40 3. 연결재무제표 주석.42 1. 지배기업의 개요 (연결) .42 2. 연결재무제표 작성기준 및 중요한 회계정책 (연결).42 3. 중요한 판단과 추정 불확실성의 주요 원천 (연결) .43 4. 영업부문 (연결) .44 5. 범주별 금융상품 (연결) .46 6. 공정가치 (연결) .48 7. 공정가치측정금융자산 (연결).51 8. 관계기업투자주식 (연결).54 9. 종속기업 (연결) .58 10. 유형자산 (연결) .67 11. 무형자산 (연결) .68 12. 투자부동산 (연결).69 13. 리스 (연결) .71 14. 리스부채 (연결) .73 15. 금융리스채권 (연결).74 16. 충당부채 (연결) .76 17. 재무위험관리 (연결).79 18. 우발채무와 약정사항 (연결) .81 19. 납입자본 (연결) .83 20. 이익잉여금 (연결).84 21. 기타자본항목 (연결).85 22. 매출액 (연결).86 23. 판매비와 관리비 (연결).89 24. 기타수익 및 기타비용 (연결).91 25. 금융수익 및 금융비용 (연결).92 26. 법인세비용 (연결).93 27. 현금흐름표 (연결).94 28. 주당이익 (연결) .96 29. 특수관계자 (연결).97 30. 주식기준보상 (연결).109 4. 재무

Chunking을 수행합니다.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name='gpt-4o-mini',
    chunk_size=1000, chunk_overlap=200)

chunks = text_splitter.split_documents(all_papers)
print(len(chunks))

740


Embedding 모델을 구성합니다.

In [10]:
from langchain_openai import OpenAIEmbeddings
openai_embeddings = OpenAIEmbeddings(model='text-embedding-3-large', chunk_size=100)

ChromaDB를 구성합니다.

In [11]:
from langchain_chroma import Chroma

Chroma().delete_collection() # (메모리에 저장하는 경우) 기존 데이터 삭제

# DB 구성하기
db = Chroma(embedding_function=openai_embeddings,
            persist_directory="./evaluation",
            collection_metadata={'hnsw:space':'l2'},
            )

DB에 document를 추가합니다.

In [12]:
db.add_documents(chunks)

['14621a59-892a-4e10-ac8e-7df0f569b3c2',
 'd5f66c1d-77c4-4500-af58-b80d4c2addd0',
 'f86457ae-eaeb-491d-ab0a-c43a1514e927',
 'c9604467-4524-4f9b-b921-ba11c7c377b9',
 '58b569ff-c469-4084-9ef9-87639f0de059',
 '1fa07958-3242-4659-85f9-8dd52f2a7e40',
 '1fb67761-bab6-4b4c-ab6e-712e20e04678',
 'c540e92b-a6ab-4464-b885-b6bb8d45a6f8',
 'a309652b-2d84-43cd-bf26-15b6f91a2657',
 '3dec8683-54ce-4576-af56-67f40d284d04',
 '898766e6-b9c7-4f05-92d5-382ffacd0841',
 'f0224abe-1074-43d9-936e-be2a8df1de0a',
 '4eeb8d4b-aa28-4b50-9623-fea182d6b537',
 'a3ef0d01-188f-49b9-97b4-468cd9e2ac10',
 '523a54f3-ecb3-4ac6-8ea6-7521c3a911b3',
 'b3541f07-ffc7-41cc-a66d-0153d6d9952e',
 '16c46a29-3eb9-47aa-85a2-dd1ef67d6791',
 '3ccf2d5f-90fc-43a5-8d85-c0c871da6457',
 '48812889-aadb-4805-b916-60b0531755fa',
 'c11d63d1-e3dc-415a-b6c3-a0f201a4f09d',
 'f2da2083-2c43-4525-9af0-1885d1436f07',
 '73a6b871-741c-47c8-9ecb-c492a3d32c9b',
 '8f464169-cacd-42bf-85ec-e06a2f156200',
 'feacb5d4-dcba-4378-bbe0-6f7084da97af',
 '5a557f49-81dc-

db로부터 retriever를 구성합니다.

In [13]:
retriever = db.as_retriever(search_kwargs={'k':5})
# Chunk Size * K = Context 글자
# 1000      *  5 = 5000 토큰 

RAG를 위한 간단한 프롬프트를 작성합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate([
    ("system", '''당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.'''),
("human",'''
Context: {context}
---
Question: {question}''')])

prompt.pretty_print()

================================ System Message ================================

당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.

================================ Human Message =================================


Context: {context}
---
Question: {question}


RAG를 수행하기 위한 Chain을 만듭니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_community.output_parsers import StrOutputParser

def format_docs(docs):
    return " \n---\n ".join([doc.page_content+ '\n' for doc in docs])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [35]:
rag_chain_from_docs = (
    prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_ref = RunnableParallel(
    context = retriever | format_docs, question = RunnablePassthrough()
).assign(answer=rag_chain_from_docs)
# context, question, answer

# RAGAS 사용하기


RAGAS 는 다양한 메트릭을 통한 RAG의 성능 평가를 지원합니다.


각각의 메트릭은 아래의 링크에서 확인할 수 있습니다.

https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/

시트에서 평가 데이터셋을 불러옵니다.   
Question/Ground Truth의 구성입니다.

In [18]:
# Best: Retrieval의 Ground Truth, Generation의 Ground Truth가 모두 존재

import pandas as pd
df = pd.read_csv('./eval_dataset.csv', encoding='cp949')
eval_dataset = df.to_dict('list')
eval_dataset

{'questions': ['2022~2024년 삼성SDS의 매출 구성을 IT서비스와 물류 관점에서 비교하고, 이 변화가 의미하는 전략적 시사점을 설명하시오.',
  '삼성SDS가 ‘클라우드 기업’으로 전환한다고 밝힌 바 있는 전략을, 제품·플랫폼·인프라·交부문 운영모델 측면에서 통합적으로 설명하시오.',
  '물류 부문에서 삼성SDS가 디지털 포워딩 경쟁력을 어떻게 구축했는지, 플랫폼 진화와 글로벌 네트워크 관점에서 설명하시오.',
  '회사가 서비스 가격표를 제시하지 않는 이유와 실제 가격 결정 방식은 무엇이며, 이는 IT서비스·물류 사업의 어떤 특성과 연결되는가?',
  '경기변동이 IT서비스와 물류에 미치는 영향과, 삼성SDS가 그 민감도를 낮추기 위해 취하는 대응을 설명하시오.',
  'SRM 분야에서의 인수·출시 활동이 삼성SDS의 SaaS 전략과 글로벌 확장에 어떤 의미가 있는지 설명하시오.',
  '인력·시설·보안 측면에서 삼성SDS의 운영 기반과 이것이 서비스 신뢰성에 주는 함의를 설명하시오.',
  '재무 안정성 측면에서 회사의 공시 내용을 종합하되, 상장 이력과 조달 구조까지 연결해 설명하시오.',
  '삼성SDS의 2024년 3분기말(9월 30일) 연결 기준 부채비율을 재무상태표 수치로 검증하고, 전기말(2023년 12월 31일)과 비교해 변화 요인을 설명하시오.',
  '2024년 3분기 누적과 2024 사업연도(사업보고서 기준)에서 삼성SDS의 희석주당이익이 기본주당이익과 동일한 이유는 무엇이며, 각 기간의 기본주당이익 산출 근거를 제시하시오.',
  '2024년 삼성SDS의 주식기준보상(엠로 포함) 현황을 수량, 행사가격, 만기, 비용 인식, 자본변동 영향까지 종합해 설명하시오.',
  '온실가스 배출권과 배출부채에 대해, 2024년 중 배출권 수량·장부금액 변동과 기말 배출부채(충당부채) 인식, 담보 제공 여부를 함께 설명하시오.',
  '특수관계자와의 배당거래는 2024년에 어떻게 이루어졌으며, 연결 현금흐름에서의 배당금 지급과 

구성된 RAG 체인을 이용해, RAGAS의 Evaluate 형태로 변환합니다.

In [19]:
questions, ground_truths = eval_dataset['questions'], eval_dataset['ground_truths']

In [20]:
for i in range(len(questions)):
    print(f'#{i}')
    print(f'Question: {questions[i]}\n')
    print(f'Ground Truth: {ground_truths[i]}\n')
    print('-----------')

#0
Question: 2022~2024년 삼성SDS의 매출 구성을 IT서비스와 물류 관점에서 비교하고, 이 변화가 의미하는 전략적 시사점을 설명하시오.

Ground Truth: 공시자료에 따르면 전체 매출에서 IT서비스와 물류 비중은 2022년 34.6% 대 65.4%, 2023년 46.0% 대 54.0%, 2024년 46.3% 대 53.7%로 나타났습니다. 특히 IT서비스 내에서도 클라우드 매출이 2022년 1,162,676백만원(6.7%) → 2023년 1,880,680백만원(14.2%) → 2024년 2,323,476백만원(16.8%)로 빠르게 확대된 반면, SI와 ITO는 정체 또는 소폭 감소했습니다. 이는 2021~2022년 물류 운임 급등 효과가 둔화되며 물류 비중이 정상화되는 가운데, 회사가 클라우드·SaaS·생성형 AI 중심의 포트폴리오 전환으로 성장축을 IT로 재정렬하고 있음을 보여줍니다. 전략적으로는 SCP 기반 CSP·MSP 역량, Brity Works·Caidentia 등 엔터프라이즈 SaaS, FabriX·Brity Copilot·GPUaaS 같은 생성형 AI 상품을 앞세워 고성장·고부가의 클라우드 매출 비중을 키우고, 물류는 Cello·Cello Square로 디지털포워딩과 운영효율화 경쟁력을 강화해 양축의 균형 성장을 지향하는 방향으로 해석됩니다.

-----------
#1
Question: 삼성SDS가 ‘클라우드 기업’으로 전환한다고 밝힌 바 있는 전략을, 제품·플랫폼·인프라·交부문 운영모델 측면에서 통합적으로 설명하시오.

Ground Truth: 제품·플랫폼 측면에서 회사는 SCP를 전개하며 CSP·MSP·SaaS를 End-to-End로 제공하고, 협업·업무혁신 솔루션 Brity Works, 글로벌 SRM SaaS ‘Caidentia’(엠로 지분 인수 기반), 공급망·인사 등 Nexprime/Nextprime 기반 솔루션을 갖추고 있습니다. 생성형 AI 측면에서는 기업용 AI 플랫폼 FabriX, 협업AI인 Brity Co

RAG 체인을 실행해, 테스트 문제에 대한 답변을 생성합니다.

In [ ]:
rag_chain_from_docs = (
    prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_ref = RunnableParallel(
    raw_context = retriever, question = RunnablePassthrough()
).assign(context = lambda x: format_docs(x['raw_context'])).assign(answer=rag_chain_from_docs)

rag_chain_with_ref.invoke("이 보고서는 어느 회사 꺼야?")
# raw_context, question, context, answer

{'raw_context': [Document(id='05034748-0d1c-40dc-a021-005fa132382f', metadata={'source': 'reports\\사업보고서(2025.03.11).pdf', 'index': 2}, page_content='배출부채 (연결).162 38. 사업결합 (연결) .164 39. 주식기준보상 (연결).166 4. 재무제표.170 4-1. 재무상태표.170 4-2. 손익계산서.171 4-3. 포괄손익계산서.172 4-4. 자본변동표.172 4-5. 현금흐름표.172 5. 재무제표 주석.175 1. 당사의 개요.175 2. 별도재무제표 작성기준 및 중요한 회계정책.175 3. 중요한 판단과 추정 불확실성의 주요 원천.199 4. 현금및현금성자산.201 5. 사용이 제한되거나 담보로 제공된 금융자산.201 6. 범주별 금융상품.202 7. 금융상품의 공정가치.205 8. 매출채권 및 미수금.210 9. 재고자산.212 10. 공정가치측정금융자산.213 11. 관계기업투자주식.216 12. 종속기업투자주식.216 13. 유형자산.219 14. 무형자산.220 15. 리스.225 16. 리스부채.226 17. 금융리스채권.228 18. 충당부채.229 19. 퇴직급여제도.231 20. 우발채무와 약정사항.235 21. 납입자본.238 22. 이익잉여금.238 23. 기타자본항목.240 24. 매출액.241 25. 비용의 성격별 분류 등.244 26. 판매비와 관리비.245 27. 기타수익 및 기타비용.247 28. 금융수익 및 금융비용.248 29. 법인세비용.249 30. 현금흐름표.252 31. 재무위험관리.256 32. 특수관계자.262 33. 주당이익.273 34. 재무제표의 승인.274 35. 온실가스 배출권과 배출부채.274 6. 배당에 관한 사항.276 7. 증권의 발행을 통한 자금조달에 관한 사항.278 7-1. 증권의 발행을 통한 자금조달 실적.278 7-2. 증권의 발행을 통해 조달된 자금의 사용실

In [48]:
from tqdm import tqdm

results = rag_chain_with_ref.batch(questions)

dataset=[]

for result, reference in tqdm(zip(results,ground_truths)):
    dataset.append(
        {
            "user_input":result['question'],
            "retrieved_contexts":[doc.page_content for doc in result['raw_context']],
            "response":result['answer'],
            "reference":reference
        }
    )    

20it [00:00, 19017.47it/s]


In [ ]:
# # [질문, 검색결과, 답변, 정답(레퍼런스)]
# for query,reference in tqdm(zip(questions,ground_truths)):

#     relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
#     response = rag_chain.invoke(query)
#     dataset.append(
#         {
#             "user_input":query,
#             "retrieved_contexts":relevant_docs,
#             "response":response,
#             "reference":reference
#         }
#     )


20it [01:08,  3.41s/it]


In [49]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset)
evaluation_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=20)

RAGAS는 LLM을 이용해 정답과 답변을 개별 Claim(주장)으로 분할합니다.

이후, `LLMContextRecall`, `Faithfulness`, `FactualCorrectness` 등의 다양한 메트릭을 통해 RAG 파이프라인의 성능을 평가합니다.   
LLM 기반의 방법이므로 평가 LLM의 선정이 중요하며, 절대 수치보다는 상대적 비교가 효과적입니다.


- Context Recall: 정답의 Claim들이 모두 검색됐는가
- Faithfulness : 답변의 Claim이 얼마나 검색 결과에 근거했는가
- Factual Correctness : 정답과 답변의 Claim이 얼마나 일치하는가
- Bleu Score : 정답과 답변 키워드가 얼마나 일치하는가
- Semantic Sim : 정답 답변 임베딩이 얼마나 가까운가   


1) Context Recall: 검색 결과 - 정답

낮은 값 --> Retrieval 실패

2) Faithfulness: 검색 결과 - 답변

낮은 값 --> LLM의 성능 문제

3) Factual Correctness: 정답 - 답변

F1 Score 방식


In [50]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
from ragas.metrics import BleuScore, SemanticSimilarity

# https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/

# 평가자 LLM
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))

# 평가자 Embedding
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model = 'text-embedding-3-large'))
semantic_scorer = SemanticSimilarity(embeddings = evaluator_embeddings)


result = evaluate(dataset=evaluation_dataset,
                  metrics=[BleuScore(), LLMContextRecall(), semantic_scorer, Faithfulness(), FactualCorrectness()],
                  llm=evaluator_llm,
                  embeddings = evaluator_embeddings
                  )
result

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

{'bleu_score': 0.0397, 'context_recall': 0.6558, 'semantic_similarity': 0.5248, 'faithfulness': 0.5666, 'factual_correctness(mode=f1)': 0.2140}

In [53]:
result.scores

[{'bleu_score': 0.029189453645337295,
  'context_recall': 1.0,
  'semantic_similarity': 0.7769170222691305,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(0.45)},
 {'bleu_score': 0.028848167261307532,
  'context_recall': 0.8,
  'semantic_similarity': 0.6700853643309139,
  'faithfulness': 0.9444444444444444,
  'factual_correctness(mode=f1)': np.float64(0.36)},
 {'bleu_score': 0.029809077846520732,
  'context_recall': 1.0,
  'semantic_similarity': 0.8510903628726033,
  'faithfulness': 0.9333333333333333,
  'factual_correctness(mode=f1)': np.float64(0.6)},
 {'bleu_score': 0.023901021968803136,
  'context_recall': 1.0,
  'semantic_similarity': 0.7558906952664655,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(0.72)},
 {'bleu_score': 0.015330462064343475,
  'context_recall': 1.0,
  'semantic_similarity': 0.19665467940885206,
  'faithfulness': 0.0,
  'factual_correctness(mode=f1)': np.float64(0.0)},
 {'bleu_score': 0.0024510303821316144,
  'context_r

In [54]:
result.to_pandas()

,user_input,retrieved_contexts,response,reference,bleu_score,context_recall,semantic_similarity,faithfulness,factual_correctness(mode=f1)
0,"2022~2024년 삼성SDS의 매출 구성을 IT서비스와 물류 관점에서 비교하고, ...",[더욱 확대되고 있습니다. 특 히 IT서비스 산업은 고급인력 중심의 지식기반사업으로...,2022년부터 2024년까지 삼성SDS의 매출 구성은 IT서비스와 물류 부문에서 다...,공시자료에 따르면 전체 매출에서 IT서비스와 물류 비중은 2022년 34.6% 대 ...,0.029189,1.000000,0.776917,1.000000,0.45
1,"삼성SDS가 ‘클라우드 기업’으로 전환한다고 밝힌 바 있는 전략을, 제품·플랫폼·인...",[높입니다. 당사는 클라우드 기업으로의 전환을 위해 클라우드 기술력 강화와 전문인력...,"삼성SDS는 ‘클라우드 기업’으로의 전환 전략을 제품, 플랫폼, 인프라, 그리고 운...",제품·플랫폼 측면에서 회사는 SCP를 전개하며 CSP·MSP·SaaS를 End-to...,0.028848,0.800000,0.670085,0.944444,0.36
2,"물류 부문에서 삼성SDS가 디지털 포워딩 경쟁력을 어떻게 구축했는지, 플랫폼 진화와...",[한계에 직면하였습니 다. 이에 따라 최근 국내 IT서비스 업체는 자체 솔루션에 대...,물류 부문에서 삼성SDS는 Cello와 Cello Square라는 최신 IT기반의 ...,삼성SDS는 통합물류 플랫폼 Cello와 디지털 물류 플랫폼 Cello Square...,0.029809,1.000000,0.851090,0.933333,0.60
3,"회사가 서비스 가격표를 제시하지 않는 이유와 실제 가격 결정 방식은 무엇이며, 이는...",[데이터센터 인프라 서비 스 운영으로 구분할 수 있습니다. ① 애플리케이션 아웃소싱...,"회사가 서비스 가격표를 제시하지 않는 이유는, 고객의 상황과 요구사항에 따라 맞춤형...",공시자료는 IT서비스(SI·ITO·클라우드)와 물류가 개인용 규격 상품이 아니라 고...,0.023901,1.000000,0.755891,1.000000,0.72
4,"경기변동이 IT서비스와 물류에 미치는 영향과, 삼성SDS가 그 민감도를 낮추기 위해...","[및 방법 대화방에 초대받은 사용자가 대화방의 이전 대화 내역을 조회할 수 있으며,...",정보가 부족하여 답변할 수 없습니다.,"IT서비스는 기업 설비투자와 연동돼 경기 둔화 시 투자 보류 영향이 있으나, 클라우...",0.015330,1.000000,0.196655,0.000000,0.00
5,SRM 분야에서의 인수·출시 활동이 삼성SDS의 SaaS 전략과 글로벌 확장에 어떤...,"[(65.4%) 합계 6,616,222 (100.0%) 6,691,688 (100....",정보가 부족하여 답변할 수 없습니다.,회사는 2023년 2분기에 국내 1위 SRM 기업 엠로의 지분 인수를 완료해 핵심 ...,0.002451,1.000000,0.143968,0.000000,0.00
6,인력·시설·보안 측면에서 삼성SDS의 운영 기반과 이것이 서비스 신뢰성에 주는 함의...,[(홈페이지) http://www.samsungsds.com 작 성 책 임 자 : ...,"삼성SDS는 인력, 시설, 보안 측면에서 안정적이고 신뢰성 높은 운영 기반을 갖추고...","2024년 말 기준 연결 인력은 약 25,666명으로, IT서비스 16,798명, ...",0.074749,0.800000,0.803426,0.647059,0.65
7,"재무 안정성 측면에서 회사의 공시 내용을 종합하되, 상장 이력과 조달 구조까지 연결...",[목 차 반 기 보 고 서.1 【 대표이사 등의 확인 】.2 I. 회사의 개요.3 ...,정보가 부족하여 답변할 수 없습니다.,"삼성SDS는 2014년 11월 14일 유가증권시장에 상장했으며, 단기 조달 수단인 ...",0.012977,0.500000,0.152661,1.000000,0.00
8,삼성SDS의 2024년 3분기말(9월 30일) 연결 기준 부채비율을 재무상태표 수치...,"[814,126 818,854 712,749 ㆍ사용권자산 916,317 821,71...",삼성SDS의 2024년 3분기말(9월 30일) 연결 기준 부채비율은 다음과 같이 계...,"2024년 3분기말 연결 기준 부채총계는 3,409,626,256천원, 자본총계는 ...",0.012962,1.000000,0.700314,0.916667,0.26
9,2024년 3분기 누적과 2024 사업연도(사업보고서 기준)에서 삼성SDS의 희석주...,[따라 작성되었습니다. 나. 요약 별도재무정보 *삼성에스디에스 주식회사 매출액 10...,정보가 부족하여 답변할 수 없습니다.,삼성SDS는 당기 및 전기에 희석증권을 보유하지 않아 희석주당이익이 기본주당이익과 ...,0.007871,0.500000,0.144808,0.000000,0.00


In [55]:
# dict로 변경하기(복잡..)
import ast

result_str = str(result)
result_dict = ast.literal_eval(result_str)
result_dict

{'bleu_score': 0.0397,
 'context_recall': 0.6558,
 'semantic_similarity': 0.5248,
 'faithfulness': 0.5666,
 'factual_correctness(mode=f1)': 0.214}

# [실습] 다양한 파이프라인 파라미터 수정하기

현재 파이프라인에는 매우 다양한 조절 가능한 파라미터가 있습니다.   

각각의 파라미터를 수정하여, 성능 변화를 확인해 보세요.

Ex)
- Chunk의 크기/개수 늘리기   
- 모델 더 저렴한 모델로 바꾸기   
-

In [63]:
MODEL_NAME='gpt-5'
TEMPERATURE=0
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 5

# CSV 파일 경로 설정
csv_path = "experiments/experiment_log.csv"
os.makedirs("experiments", exist_ok=True)


def RAG_pipeline():

    dataset = []

    llm = ChatOpenAI(model=MODEL_NAME, temperature = TEMPERATURE, max_tokens = 4096)
    Chroma().delete_collection() # 메모리 기존 데이터 삭제
    

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    print('## DB 구성 중...')
    

    chunks = text_splitter.split_documents(all_papers)
    print(f'## Chunks 생성 완료... 총 {len(chunks)} 개 청크')




    db = Chroma(embedding_function=openai_embeddings,
                collection_metadata={'hnsw:space':'l2'},
                )
    for i in tqdm(range(0, len(chunks), 50)):
        db.add_documents(chunks[i:min(i+50, len(chunks))])

    print('## DB 구성 완료...')


    retriever = db.as_retriever(search_kwargs={'k':TOP_K})


    def format_docs(docs):
        return " \n---\n ".join([f'[출처]:{doc.metadata['source']}\n[내용]: {doc.page_content}\n' for doc in docs])
    
    rag_chain_from_docs = (
    prompt
    | llm
    | StrOutputParser()
    )


    rag_chain_with_ref = RunnableParallel(
        raw_context = retriever, question = RunnablePassthrough()
    ).assign(context = lambda x: format_docs(x['raw_context'])).assign(answer=rag_chain_from_docs)



    results = rag_chain_with_ref.batch(questions)

    dataset=[]

    for result, reference in tqdm(zip(results,ground_truths)):
        dataset.append(
            {
                "user_input":result['question'],
                "retrieved_contexts":[doc.page_content for doc in result['raw_context']],
                "response":result['answer'],
                "reference":reference
            }
        )    

    print('## 답변 생성 완료...')
    evaluation_dataset = EvaluationDataset.from_list(dataset)

    result = evaluate(dataset=evaluation_dataset,
                    metrics=[BleuScore(), LLMContextRecall(), semantic_scorer, Faithfulness(), FactualCorrectness()],
                    llm=evaluator_llm,
                    embeddings = evaluator_embeddings
                    )

    result_str = str(result)
    result_dict = ast.literal_eval(result_str)
    return result_dict


실험 시작 시간과 정보를 저장할 수 있습니다.

In [57]:
import csv
import datetime
import os
def run_experiment():
    result = RAG_pipeline()
    timestamp = datetime.datetime.now().isoformat()
    row = {
        "timestamp": timestamp,
        "model_name": MODEL_NAME,
        "temperature": TEMPERATURE,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "top_k": TOP_K,
        **result
    }
    # 파일이 없으면 헤더 추가
    write_header = not os.path.exists(csv_path)

    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    print(f"[{timestamp}] {MODEL_NAME} | temp={TEMPERATURE}, chunk={CHUNK_SIZE}/{CHUNK_OVERLAP}, top_k={TOP_K} | results={result}")


In [58]:
MODEL_NAME='gpt-4.1-nano'
TEMPERATURE=0
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 5

run_experiment()

## DB 구성 중...
## Chunks 생성 완료... 총 1337 개 청크


100%|██████████| 27/27 [00:56<00:00,  2.11s/it]


## DB 구성 완료...


20it [00:00, 128070.35it/s]


## 답변 생성 완료...


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

[2025-08-30T15:14:11.118123] gpt-4.1-nano | temp=0, chunk=1000/200, top_k=5 | results={'bleu_score': 0.055, 'context_recall': 0.5692, 'semantic_similarity': 0.5172, 'faithfulness': 0.5102, 'factual_correctness(mode=f1)': 0.253}


In [59]:
MODEL_NAME='gpt-4.1-nano'
TEMPERATURE=0
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 10

run_experiment()

## DB 구성 중...
## Chunks 생성 완료... 총 1337 개 청크


100%|██████████| 27/27 [00:51<00:00,  1.91s/it]


## DB 구성 완료...


20it [00:00, 122104.92it/s]


## 답변 생성 완료...


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

[2025-08-30T15:17:22.657510] gpt-4.1-nano | temp=0, chunk=1000/200, top_k=10 | results={'bleu_score': 0.0556, 'context_recall': 0.7483, 'semantic_similarity': 0.6135, 'faithfulness': 0.6623, 'factual_correctness(mode=f1)': 0.294}


In [60]:
MODEL_NAME='gpt-4.1-nano'
TEMPERATURE=0
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 400
TOP_K = 5

run_experiment()

## DB 구성 중...
## Chunks 생성 완료... 총 668 개 청크


100%|██████████| 14/14 [00:35<00:00,  2.52s/it]


## DB 구성 완료...


20it [00:00, 167437.29it/s]


## 답변 생성 완료...


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

[2025-08-30T15:19:49.027079] gpt-4.1-nano | temp=0, chunk=2000/400, top_k=5 | results={'bleu_score': 0.0421, 'context_recall': 0.6392, 'semantic_similarity': 0.5016, 'faithfulness': 0.4118, 'factual_correctness(mode=f1)': 0.208}


In [61]:
MODEL_NAME='gpt-4.1-nano'
TEMPERATURE=0
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 400
TOP_K = 10

run_experiment()

## DB 구성 중...
## Chunks 생성 완료... 총 668 개 청크


100%|██████████| 14/14 [00:40<00:00,  2.89s/it]


## DB 구성 완료...


20it [00:00, 125766.24it/s]


## 답변 생성 완료...


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

[2025-08-30T15:22:37.118243] gpt-4.1-nano | temp=0, chunk=2000/400, top_k=10 | results={'bleu_score': 0.0465, 'context_recall': 0.8475, 'semantic_similarity': 0.578, 'faithfulness': 0.6018, 'factual_correctness(mode=f1)': 0.316}


다양한 기법을 위 코드에서 적용하여 성능을 높일 수 있습니다.

In [62]:
MODEL_NAME='gpt-4.1-mini'
TEMPERATURE=0
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 400
TOP_K = 10

run_experiment()

## DB 구성 중...
## Chunks 생성 완료... 총 668 개 청크


100%|██████████| 14/14 [00:38<00:00,  2.74s/it]


## DB 구성 완료...


20it [00:00, 118650.75it/s]


## 답변 생성 완료...


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

[2025-08-30T15:28:18.083208] gpt-4.1-mini | temp=0, chunk=2000/400, top_k=10 | results={'bleu_score': 0.0593, 'context_recall': 0.8183, 'semantic_similarity': 0.7289, 'faithfulness': 0.9059, 'factual_correctness(mode=f1)': 0.429}


In [64]:
MODEL_NAME='gpt-5-mini'
TEMPERATURE=0
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 400
TOP_K = 10

run_experiment()

## DB 구성 중...
## Chunks 생성 완료... 총 668 개 청크


100%|██████████| 14/14 [00:37<00:00,  2.68s/it]


## DB 구성 완료...


20it [00:00, 153076.79it/s]


## 답변 생성 완료...


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

[2025-08-30T15:34:36.372959] gpt-5-mini | temp=0, chunk=2000/400, top_k=10 | results={'bleu_score': 0.0325, 'context_recall': 0.8392, 'semantic_similarity': 0.6444, 'faithfulness': 0.8142, 'factual_correctness(mode=f1)': 0.443}
